# Поиск аномалий

Методы обнаружения аномалий, как следует из названия, позволяют находить необычные объекты в выборке. Но что такое "необычные" и совпадает ли это определение у разных методов?

Начнём с поиска аномалий в текстах: научимся отличать вопросы о программировании от текстов из 20newsgroups про религию.

Подготовьте данные: в обучающую выборку возьмите 20 тысяч текстов из датасета Stack Overflow, а тестовую выборку сформируйте из 10 тысяч текстов со Stack Overflow и 100 текстов из класса soc.religion.christian датасета 20newsgroups (очень пригодится функция `fetch_20newsgroups(categories=['soc.religion.christian'])`). Тексты про программирование будем считать обычными, а тексты про религию — аномальными.

In [1]:
import sklearn as sk
from sklearn.model_selection import train_test_split

In [2]:
from sklearn.datasets import fetch_20newsgroups

data = fetch_20newsgroups(
    categories=['soc.religion.christian'],
    shuffle=True,
    random_state=42,

)

texts = data.data[:100]


In [3]:
import pandas as pd
import numpy as np

df = pd.read_parquet("post_questions_test_000000000000.parquet")
df = df["body"].reset_index(drop = True)
x_train, x_test = train_test_split(df, train_size = 20000, test_size = 10000, random_state = 52, shuffle = True)
y_train = pd.Series([0] * len(x_train))
y_test = pd.Series([0] * len(x_test) + [1] * len(texts))
texts = pd.Series(texts)
x_test = pd.concat([x_test, texts])
x_train = x_train.reset_index(drop=True)
x_test = x_test.reset_index(drop = True)

perm = np.random.permutation(len(x_test))
x_test = x_test.iloc[perm]
y_test = y_test.iloc[perm]
x_train = x_train.reset_index(drop=True)
x_test = x_test.reset_index(drop = True)
print(x_test[10000:10100])

10000    <p>I want to add a new button beside slide sho...
10001    <p>I have a rails app hosted on heroku and a m...
10002    <p>I have an AsyncTask running and in this thr...
10003    <p>I want to get WebView height knowing it's c...
10004    <p>We monitor our Spring Boot apps with the Sp...
                               ...                        
10095    <p>I'm using Twitter Bootstrap right now ver 2...
10096    <p>I am trying to convert decimal Nos. into bi...
10097    <p>I have followed this (IIS Windows Container...
10098    <p>I've been looking into locale aware number ...
10099    <p>I want to make sure what screenshot need to...
Length: 100, dtype: object


**(1 балл)**

Проверьте качество выделения аномалий (pre и rec на тестовой выборке, если считать аномалии положительным классов, а обычные тексты — отрицательным) для IsolationForest. В качестве признаков используйте TF-IDF, где словарь и IDF строятся по обучающей выборке. Не забудьте подобрать гиперпараметры.

In [4]:
print(x_test)

0        <p>I am trying to create a discord bot, howeve...
1        <p>When I click on 'add to cart' button then i...
2        <p>I have a <a href="http://docs.sencha.com/ex...
3        <p>I have 3 entities:</p>\n\n<ul>\n<li>First o...
4        <p>I have a Telerik RadScheduler which display...
                               ...                        
10095    <p>I'm using Twitter Bootstrap right now ver 2...
10096    <p>I am trying to convert decimal Nos. into bi...
10097    <p>I have followed this (IIS Windows Container...
10098    <p>I've been looking into locale aware number ...
10099    <p>I want to make sure what screenshot need to...
Length: 10100, dtype: object


In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import IsolationForest
from sklearn.metrics import precision_score, recall_score

vec = TfidfVectorizer(max_features=5000) 
x_train_v = vec.fit_transform(x_train)
x_test_v  = vec.transform(x_test)
iso = IsolationForest(
    n_estimators=100,
    max_samples='auto',
    contamination=0.01,
    random_state=42
)
iso.fit(x_train_v)
y_pred = iso.predict(x_test_v)




In [6]:
print(len(y_pred))
for i in range(len(x_test)):
    if(y_pred[i] == -1):

        print(x_test[i][:20])


10100
<p>I'm writing a web
<p>I've been searchi
<p>I would like to d
<p>I am trying to wr
From: jsledd@ssdc.sa
<p>I have the follow
<p>Sorry for the rat
<p>I'm working on a 
<p>I'm running into 
<p>So I have two tab
<p>i am developing a
<p>I'm trying to mak
<p>I have studied <a
<p>Here's the proble
<p>I am creating an 
<p>I am currently wo
<p>On my site every 
<p>My aim is to find
From: littlejs@nextw
<p>I'm building a do
<p>I an using <a hre
<p>I am currently bu
<p>I am working on a
From: pharvey@quack.
From: rjb@akgua.att.
<p>I have an ASP.NET
<p>I've been reading
<p>I read some threa
<p>I've reviewed sim
<p>I am developing a
<p>I am getting the 
<p>We are porting ou
<p>I've been reading
<p>What is the best 
<p>I'm working on am
<p>I want to play an
<p>he ppl!</p>

<p>i
<p>I am using NSFetc
<p>As part of some e
<p>Lots of pages in 
<pre><code>def compr
From: rayssd!esther@
<p>I'm trying to wri
<p>I'm using jQuery 
<p><strong>strong te
<p>I am trying to im
<p>I hope someone he
<p>We a

In [7]:

y_pred = (y_pred == -1).astype(int)
# print(y_pred)
pre = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
print(pre, rec)

0.09482758620689655 0.11


**(5 баллов)**

Скорее всего, качество оказалось не на высоте. Разберитесь, в чём дело:
* посмотрите на тексты, которые выделяются как аномальные, а также на слова, соответствующие их ненулевым признакам
* изучите признаки аномальных текстов
* посмотрите на тексты из обучающей выборки, ближайшие к аномальным; действительно ли они похожи по признакам?

Сделайте выводы и придумайте, как избавиться от этих проблем. Предложите варианты двух типов: (1) в рамках этих же признаков (но которые, возможно, будут считаться по другим наборам данных) и методов и (2) без ограничений на изменения. Реализуйте эти варианты и проверьте их качество.

In [8]:
an_idx = np.where(y_pred == 1)[0]
print(len(y_pred))
for i in an_idx[:5]:
    
    print(x_test[i][0:300])
    pass


10100
<p>I'm writing a web app that needs to use several ansi C functions to crunch data. Another language, probably Java, will call the main C function to trigger the process. I want to return a value from the main C function for the Java script to evaluate success or not. I expect C's use of <code>exit(
<p>I've been searching and trying different things for over an hour. I've found many articles that are basically what's happening to me, but nothing I'm finding/trying is fixing the 404. Angular is converting my $http.post to method OPTIONS. This is causing my node/express route to fail because ther
<p>I would like to debug client side applications by manually creating a JSON data stream. I was hoping that Google Chrome's debug console would be able to do this.</p>

<p>For example, consider the following case. I want to test how a list population code segment would work. Let's say I have item o
<p>I am trying to write a function that lets me insert a value into the firebird database. 

In [9]:
feature_names = np.array(vec.get_feature_names_out())

def top_words(row, topn=10):
    arr = row.toarray().ravel()
    idx = np.argsort(arr)[-topn:]
    return feature_names[idx]

for i in an_idx[:5]:
    print(top_words(x_test_v[i]))

['return' 'to' 'function' 'exit' 'main' 'prototype' 'the' 'quot'
 'statements' 'code']
['stack' 'img' 'png' 'express' 'imgur' 'alt' 'enter' 'description' 'post'
 'route']
['log' 'to' 'this' 'like' 'debug' 'console' 'key' 'price' 'data' 'would']
['status' 'insert' '403' 'node' 'values' 'query' 'err' 'body' 'req' 'res']
['purposes' 'it' 'your' 'edu' 'you' 'little' 'to' 'the' 'life' 'love']


In [21]:
from sklearn.neighbors import NearestNeighbors

nn = NearestNeighbors(n_neighbors=3, metric='cosine')
nn.fit(x_train_v)

dist, ind = nn.kneighbors(x_test_v[an_idx][:5])

for i, idx in enumerate(an_idx[:5]):
    print("anom:")
    print(x_test.iloc[idx][:20])
    print("\nknn:")
    for j in ind[i]:
        print("-", x_train.iloc[j][:20])
    print("="*80)


anom:
<p> I'm writing a we

knn:
- <p> I have a questio
- <p> I am trying to u
- <p> I have developed
anom:
<p> I've been search

knn:
- <p> I have problem w
- <p> I'm completely s
- <p> I have a server 
anom:
<p> I would like to 

knn:
- <p> I retrieve data 
- <p> Consider I have 
- <p> i want to access
anom:
<p> I am trying to w

knn:
- <p> I'm new to Node.
- <p> I am unsure as t
- <p> I have a simple 
anom:
From: jsledd@ssdc.sa

knn:
- <p> What are useful 
- <h2> Original Questi
- <p> I am doing a web


### Эксперимент только с изменением датасета

In [22]:
import re
import pandas as pd

def sep_html(text):
    text = re.sub(r'(<[^>]+>)', r' \1 ', text)
    
    text = re.sub(r'\s+', ' ', text)
    
    return text.strip()
    
x_train = x_train.apply(sep_html)
x_test = x_test.apply(sep_html)


# print(x_test)
# x_test

In [ ]:
vec = TfidfVectorizer(
    max_features=5000
)
x_train_v = vec.fit_transform(x_train)
x_test_v = vec.transform(x_test)
iso = IsolationForest(
    n_estimators=100,
    max_samples="auto",
    contamination=0.01,
    random_state=42,
)
iso.fit(x_train_v)
y_pred = iso.predict(x_test_v)


y_pred = (y_pred == -1).astype(int)
# print(y_pred)
pre = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
print(pre, rec)

[0 0 0 ... 0 0 0]
0.09482758620689655 0.11


### Эксперимент с любыми изменениями

In [24]:
import re
import html
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.metrics import precision_score, recall_score
from sklearn.neighbors import NearestNeighbors


def getn(text):


    code_ptn = re.compile(r'(<pre.*?>.*?</pre>|<code.*?>.*?</code>|```[\s\S]*?```)', flags=re.I)
    html_tag_ptn = re.compile(r'(<[^>]+>)')
    email_header_ptn = re.compile(r'(?m)^(From|Subject|Reply-To|Organization|In article|Lines|Date):')

    quote_ptn = re.compile(r'(?m)^(>+)\s*')
    url_ptn = re.compile(r'https?://\S+|www\.\S+')
    email_addr_ptn = re.compile(r'\b[\w\.-]+@[\w\.-]+\.\w+\b')

    s = str(text)
    s = html.unescape(s)

    codes = code_ptn.findall(s)
    n_code_blocks = len(codes)
    s = code_ptn.sub(' __CODE_BLOCK__ ', s)

    tags = html_tag_ptn.findall(s)
    n_html_tags = len(tags)
    s = html_tag_ptn.sub(' __HTML_TAG__ ', s)

    n_email_headers = sum(1 for _ in email_header_ptn.finditer(s)) 
    s = email_header_ptn.sub(' __EMAIL_HEADER__ ', s)

    n_urls = len(url_ptn.findall(s))
    s = url_ptn.sub(' __URL__ ', s)

    n_emails = len(email_addr_ptn.findall(s)) 
    s = email_addr_ptn.sub(' __EMAIL_ADDR__ ', s)


    s = re.sub(r'\s+', ' ', s).strip()

    ft = {
        # 'n_code_blocks': n_code_blocks,
        # 'n_html_tags': -n_html_tags,
        'n_email_headers': n_email_headers,
        # 'n_urls': n_urls,
        'n_emails': n_emails,
    }

    return s, ft


In [25]:
def bf(x_train, x_test):
    alls = pd.concat([x_train, x_test], ignore_index=True)

    clt = []
    ftrtr = []

    for t in alls:
        c, f = getn(t)
        clt.append(c)
        ftrtr.append(f)

    df_features = pd.DataFrame(ftrtr).fillna(0)

    n_train = len(x_train)

    clean_train = pd.Series(clt[:n_train])
    clean_test = pd.Series(clt[n_train:])

    df_train = df_features.iloc[:n_train].reset_index(drop=True)
    df_test = df_features.iloc[n_train:].reset_index(drop=True)

    return clean_train, clean_test, df_train, df_test


In [26]:

from sklearn.preprocessing import StandardScaler
from scipy import sparse

scaler = StandardScaler()


def bvct(clean_train, clean_test, df_train, df_test):
    vec = TfidfVectorizer(
        max_features=3000,
        stop_words='english',
        min_df=3,
        ngram_range=(1,2),
        sublinear_tf=True
    )

    X_tfidf_train = vec.fit_transform(clean_train)
    X_tfidf_test = vec.transform(clean_test)

    scaler = StandardScaler()
    X_num_train = scaler.fit_transform(df_train.values)
    X_num_test = scaler.transform(df_test.values)
    coef = 100000000
    X_num_train = scaler.fit_transform(df_train.values * coef)
    X_num_test  = scaler.transform(df_test.values * coef)
    X_train = sparse.hstack([X_tfidf_train, sparse.csr_matrix(X_num_train)])
    X_test = sparse.hstack([X_tfidf_test, sparse.csr_matrix(X_num_test)])
    
    X_train = sparse.hstack([sparse.csr_matrix(X_num_train)])
    X_test = sparse.hstack([sparse.csr_matrix(X_num_test)])

    return X_train, X_test, vec


In [27]:
def run(X_train, X_test, y_test):
    iso = IsolationForest(
        n_estimators=200,
        contamination=0.00990099009900990099009900990099,
        random_state=42
    )

    iso.fit(X_train)
    scores = iso.decision_function(X_test)  # array float
    # threshold_99 = np.percentile(scores, 100-0.990099009900990099009900990099)

    # y_pred = (scores >= threshold_99).astype(int)  # 1 = аномалия
    y_pred = (scores < 0).astype(int)  # 1 = аномалия

    


    pre = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)

    print("Precision:", pre)
    print("Recall:", rec)

    return y_pred


In [28]:
def showan(y_pred, clean_test, df_features_test, n=10):
    idx = np.where(y_pred == 1)[0]

    print("Tot:", len(idx))

    for i in idx[:n]:
        print("\n" + "-"*70)
        print("Text:")
        print(clean_test.iloc[i][:400])
        print("\nNumeric ft:")
        print(df_features_test.iloc[i].to_dict())


In [29]:
clean_train, clean_test, df_train, df_test = bf(x_train, x_test)

X_train, X_test, vec = bvct(
    clean_train, clean_test,
    df_train, df_test
)
# df_train['n_email_headers'] *= 10000000
# df_test['n_email_headers']  *= 10000000

y_pred = run(X_train, X_test, y_test)

showan(y_pred, clean_test, df_test, n=110)


Precision: 0.8448275862068966
Recall: 0.98
Tot: 116

----------------------------------------------------------------------
Text:
__HTML_TAG__ My problem is i am not been able to receive joined chat rooms. I am using the openfire server 3.8.2 and asmack library asmack-android-16.jar. I receive item-not-found error when i call getJoinedRooms function. though i can see the user is joined in the room from the admin console. Is it the server problem or the client problem or some issue with asmack? Please tell me if someone is ab

Numeric ft:
{'n_email_headers': 0, 'n_emails': 1}

----------------------------------------------------------------------
Text:
__HTML_TAG__ I'll run the servers in Openstack using ansible playbook. __HTML_TAG__ __HTML_TAG__ ~/ansible/roles/launch_instance/tasks/main.yml __HTML_TAG__ __CODE_BLOCK__ __HTML_TAG__ ~/ansible/roles/launch_instance/tasks/main.yml __HTML_TAG__ __CODE_BLOCK__ __HTML_TAG__ After starting the server I connect via ssh (ssh __EMAIL_ADDR__ ) a

Подготовьте выборку: удалите столбцы `['id', 'date', 'price', 'zipcode']`, сформируйте обучающую и тестовую выборки по 10 тысяч домов.

Добавьте в тестовую выборку 10 новых объектов, в каждом из которых испорчен ровно один признак — например, это может быть дом из другого полушария, из далёкого прошлого или будущего, с площадью в целый штат или с таким числом этажей, что самолётам неплохо бы его облетать стороной.

Посмотрим на методы обнаружения аномалий на более простых данных — уж на табличном датасете с 19 признаками всё должно работать как надо!

Скачайте данные о стоимости домов: https://www.kaggle.com/harlfoxem/housesalesprediction/data

In [30]:
#code here

**Задание 9. (2 балла)**

Примените IsolationForest для поиска аномалий в этих данных, запишите их качество (как и раньше, это pre и rec). Проведите исследование:

Нарисуйте распределения всех признаков и обозначьте на этих распределениях объекты, которые признаны аномальными.

In [31]:
#code here